create PDF error csv  [filename, text]
create page error csv [filename, page, type, error, image file]
create results csv [filename, ID, Q, type, status, text, image file]

Given a list of filenames #assume PDF files with scanned surveys
    Try open PDF and determine num pages
        for each page
            try rectify image and get barcode
                for each question that matches page
                    get question type
                    for each input circle
                        get true/false
                    assess answer validity
                    if OK
                        report in results
                    else
                        get question image 
                        report in results and note image
                    if written answer
                        try to use OCR to read
                            save in results
                        execept
                            save in results with link to image file

            except barcode error
                report filename and page in page error file and barcode image
            except other 
                report filename page and error in page error file
        

    except
        Report filename error in error csv file


In [1]:

import process_surveys as ps
from datetime import datetime as dt
import sys
sys.path.append(r"C:\GIT\MPG") 
import general_routines as gr
from pathlib import Path
from pypdf import PdfReader
import json
import traceback
import importlib
import re

In [2]:
#parameters
DPI = 300
MIN_CIRCLE_PIXELS = 30
MIN_TEXT_PIXELS = 300
MIN_BLUE = 100
BLUE_MARGIN = 30
COORD_FILE = r"C:\GIT\Parish-Survey\2026\Survey_coords_8OCT.json"
with open(COORD_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)
QUESTIONS = data["questions"]
ANS_TYPES = ['single','multiple_with_written_other','scale_single','multiple','single_with_written_other','written']
SINGLE_ANS = ['single','scale_single','single_with_written_other']
DELIM = "_"

In [20]:
importlib.reload(ps)

<module 'process_surveys' from 'C:\\GIT\\Parish-Survey\\2026\\process_surveys.py'>

In [3]:
dt_str = dt.now().strftime("%d-%m-%y %H%M%S")
pdf_err_file = f"results\\{dt_str} PDF errors.csv"
page_err_file = f"results\\{dt_str} Page errors.csv"
results_file = f"results\\{dt_str} Results.csv"



In [6]:
#use this routine to process all surveys all pdfs listed in list_file

pdf_list = r"C:\GIT\Parish-Survey\2026\all_pdfs_2nd_run.txt"


list_file = Path(pdf_list)

#check list_file exists
if not list_file.exists():
    print(f"ERROR: List file not found: {list_file}")

with open(list_file, "r", encoding="utf-8") as f:
    pdf_files = [line.strip() for line in f if line.strip()]
    
#get number of files and calc likely number of surverys (assuming 10 per file)        
with open(list_file, "r", encoding="utf-8") as file:
    num_surveys = 0
    for pdf_path in pdf_files:
        pdf_path = Path(pdf_path)
        if pdf_path.exists():
            reader = PdfReader(str(pdf_path))
            num_surveys += len(reader.pages)/2
        
print(f"Processing {len(pdf_files)} PDF files, {num_surveys} surveys..\n")

start = dt.now()
processed = 0
pdf_err_count = 0
page_err_count = 0 
q_err_count= 0
with open (pdf_err_file, "a") as pdf_err, \
     open (page_err_file, "a") as page_err, \
     open (results_file, "a") as results:

    for pdf_path in pdf_files:
        #Check if 
        
        #get the batch of surverys by mass from the file path
#        match = re.search(r'\\(T_[^\\]+)', pdf_path)
#        batch = match.group(1)[2:]
        batch = "extras"

        try:
            pdf_path = Path(pdf_path)
                        
            if not pdf_path.exists():
                pdf_err.write(f"{pdf_path}, does not exist")
                pdf_err_count += 1
            else:  
                reader = PdfReader(str(pdf_path))
                page_count = len(reader.pages)

                for page in range(1,page_count+1):
                    gr.progress(f"{num_surveys} surveys. pdf err: {pdf_err_count}, page err: {page_err_count}, q err: {q_err_count}", start, processed + page/2, num_surveys)  

                    #rectify
                    try:
                        png_data, survey_number, page_code = ps.rectify_survey_page(
                            pdf_file=pdf_path,
                            page_number=page,
                            json_file=COORD_FILE,
                            dpi=DPI,
                            tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe",
                        )
                        print(f"Page {page}, Survey {survey_number} OK")

                        #for each question load time
                        for q in QUESTIONS:
                            status = "OK"  
                            err_txt = []
                            save_img = "None"
                            checks = list()
                            written_ans = ""
                            img_filename = ""
                            
                            if int(page_code[1:]) == q['page']:
                                
                                for i in q['inputs']:
                                    ans_png = ps.crop_png(png_data,i['alignment_box'])
                                    if i['kind'] == 'circle':
                                        if ps.has_blue_pixels(ans_png, MIN_CIRCLE_PIXELS):
                                            checks.append(i['option_index'])
                                    elif i['kind'] == 'written':
                                        written_png = ans_png
                                        try:
                                            written_ans = ps.read_blue_text(written_png, MIN_TEXT_PIXELS, MIN_BLUE, BLUE_MARGIN, 
                                                tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe")
                                            #remove special characters
                                            if not(written_ans == None):
                                                written_ans = re.sub(r"[^A-Za-z0-9 ]", "", written_ans)
                                                save_img = "Written"
                                        except Exception as e:
                                            status = "ERROR"
                                            err_txt.append(f"Text read error {e}")
                                            save_img = "Written"
                                #check for multiple answers when not allowed
                                if q['type'] in SINGLE_ANS and len(checks) > 1:  #if multiple answers give error
                                    status = "ERROR"
                                    err_txt.append("Multiple answers")
                                    save_img = "All"
                                # determine if image needs to be saved and which one and save it
                                if not(save_img == "None"):
                                    img_filename = f"pngs\\{dt_str} {pdf_path.stem} {page} {survey_number} {q['id']}.png"
                                if save_img == "Written":
                                    with open(img_filename, "wb") as img_f:
                                        img_f.write(written_png)
                                elif save_img == "All":
                                    with open(img_filename, "wb") as img_f:
                                        q_png = ps.crop_png(png_data,q['alignment_box'])
                                        img_f.write(q_png)
                                #write results
                                output_str = dt.now().strftime("%d-%m-%y %H%M%S")
                                output_str += f", {str(pdf_path)}, {survey_number}, {page}, {q['id']}, {status}, {DELIM.join(map(str, checks))}, {written_ans}"
                                output_str += f", {DELIM.join(err_txt)}, {img_filename} \n"
                                results.write(output_str)
                                print (f"Question {q['id']} {status} {DELIM.join(err_txt)}")
                                if not(status=="OK"):
                                    q_err_count += 1
#                            else:
#                                print (f"{int(page_code[1:])} no match to {q['page']}")

                        #if P4 save batch
                        if page_code == 4:
                            output_str = dt.now().strftime("%d-%m-%y %H%M%S")
                            output_str += f", {str(pdf_path)}, {survey_number}, {page}, batch, {status}, , {batch}"
                            results.write(output_str)     


                    except ps.BarcodeReadError as error:
                        print ("barcode error")
                        # Save the barcode crop included with the error.
                        error.save_barcode_crop(f"pngs\\{dt_str} {pdf_path.stem} {page}.png")
                        page_err.write(f"{pdf_path}, page {page}, barcode error, file: {pdf_path}_{page}.png\n")
                        page_err_count += 1
                    except ps.SurveyProcessingError as error:
                        print ("survey processing error")
                        page_err.write(f"{pdf_path}, page {page}, page read error\n")
                        page_err_count += 1

            processed += page_count/2
                    
        except Exception as e:
            pdf_err.write(f"ERROR | {pdf_path} | {type(e).__name__}: {e}\n")
            pdf_err_count += 1
            print (f"Outer exception: {e}")

            error_message = (
                f"ERROR | {pdf_path} | page {page} | "
                f"question {q.get('id', 'UNKNOWN')} | "
                f"{type(e).__name__}: {e}"
            )
    
            print(error_message)
            traceback.print_exc()
            raise
#pdf_list = r"C:\GIT\Parish-Survey\2026\survey_pdfs.txt"

245.0 surveys. pdf err: 0, page err: 35, q err: 331 245.0 processed.  100.0% completed.  Elapsed: 138.2 min. Est. remaining: 0.0 min
Page 20, Survey 0120 OK
Question Q4 OK 
Question Q5a OK 
Question Q5b OK 
Question Q5c OK 
Question Q6 OK 
Question Q7 OK 
Question Q8 OK 
Question Q9 OK 
Question Q10a OK 
Question Q10b OK 
Question Q10c OK 
Question Q11 OK 
Question Q12 OK 
Question Q13 OK 
Question Q14 OK 
Question Q15 OK 
Question Q16 OK 
Question Q17 OK 
Question Q18 OK 
Question Q19 OK 


In [4]:
#use this routine to process the listed pages of the listed pdf in list file and use the supplied survey IDs and page numbers rather than deduce with OCR. 

pdf_list = r"C:\GIT\Parish-Survey\2026\14_Oct Page errors ready to be reprocessed.csv"

list_file = Path(pdf_list)

#check list_file exists
if not list_file.exists():
    print(f"ERROR: List file not found: {list_file}")

with open(list_file, "r", encoding="utf-8") as f:
    pdf_files = [line.strip() for line in f if line.strip()]

num_surveys = len(pdf_files) 
        
print(f"Processing {len(pdf_files)} PDF pages\n")

start = dt.now()
pdf_err_count = 0
page_err_count = 0 
q_err_count= 0
with open (pdf_err_file, "a") as pdf_err, \
     open (page_err_file, "a") as page_err, \
     open (results_file, "a") as results:

    for inc, page_info in enumerate(pdf_files):

        #split out comma delimeted data
        info = page_info.split(",")
        pdf_path = info[0]
        page = int(info[1])
        survey_number = info[2]
        page_code = int(info[3])
        
        #get the batch of surverys by mass from the file path

        try:
            pdf_path = Path(pdf_path)
                        
            if not pdf_path.exists():
                pdf_err.write(f"{pdf_path}, does not exist")
                pdf_err_count += 1
            else:  
                reader = PdfReader(str(pdf_path))

                gr.progress(f"{num_surveys} pages. pdf err: {pdf_err_count}, page err: {page_err_count}, q err: {q_err_count}", start, inc, num_surveys)  

                #rectify
                try:
                    png_data = ps.rectify_survey_page(
                        pdf_file=pdf_path,
                        page_number=page,
                        json_file=COORD_FILE,
                        dpi=DPI,
                        return_barcode_data = False,
                    )
                    print(f"Page {page}, Survey {survey_number} OK")

                    #for each question load time
                    for q in QUESTIONS:
                        status = "OK"  
                        err_txt = []
                        save_img = "None"
                        checks = list()
                        written_ans = ""
                        img_filename = ""
                        
                        if int(page_code) == q['page']:
                            
                            for i in q['inputs']:
                                ans_png = ps.crop_png(png_data,i['alignment_box'])
                                if i['kind'] == 'circle':
                                    if ps.has_blue_pixels(ans_png, MIN_CIRCLE_PIXELS):
                                        checks.append(i['option_index'])
                                elif i['kind'] == 'written':
                                    written_png = ans_png
                                    try:
                                        written_ans = ps.read_blue_text(written_png, MIN_TEXT_PIXELS, MIN_BLUE, BLUE_MARGIN, 
                                            tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe")
                                        #remove special characters
                                        if not(written_ans == None):
                                            written_ans = re.sub(r"[^A-Za-z0-9 ]", "", written_ans)
                                            save_img = "Written"
                                    except Exception as e:
                                        status = "ERROR"
                                        err_txt.append(f"Text read error {e}")
                                        save_img = "Written"
                            #check for multiple answers when not allowed
                            if q['type'] in SINGLE_ANS and len(checks) > 1:  #if multiple answers give error
                                status = "ERROR"
                                err_txt.append("Multiple answers")
                                save_img = "All"
                            # determine if image needs to be saved and which one and save it
                            if not(save_img == "None"):
                                img_filename = f"pngs\\{dt_str} {pdf_path.stem} {page} {survey_number} {q['id']}.png"
                            if save_img == "Written":
                                with open(img_filename, "wb") as img_f:
                                    img_f.write(written_png)
                            elif save_img == "All":
                                with open(img_filename, "wb") as img_f:
                                    q_png = ps.crop_png(png_data,q['alignment_box'])
                                    img_f.write(q_png)
                            #write results
                            output_str = dt.now().strftime("%d-%m-%y %H%M%S")
                            output_str += f", {str(pdf_path)}, {survey_number}, {page}, {q['id']}, {status}, {DELIM.join(map(str, checks))}, {written_ans}"
                            output_str += f", {DELIM.join(err_txt)}, {img_filename} \n"
                            results.write(output_str)
                            print (f"Question {q['id']} {status} {DELIM.join(err_txt)}")
                            if not(status=="OK"):
                                q_err_count += 1
#                            else:
#                                print (f"{int(page_code[1:])} no match to {q['page']}")

             


                except ps.BarcodeReadError as error:
                    print ("barcode error")
                    # Save the barcode crop included with the error.
                    error.save_barcode_crop(f"pngs\\{dt_str} {pdf_path.stem} {page}.png")
                    page_err.write(f"{pdf_path}, page {page}, barcode error, file: {pdf_path}_{page}.png\n")
                    page_err_count += 1
                except ps.SurveyProcessingError as error:
                    print ("survey processing error")
                    page_err.write(f"{pdf_path}, page {page}, page read error\n")
                    page_err_count += 1
                    raise

                    
        except Exception as e:
            pdf_err.write(f"ERROR | {pdf_path} | {type(e).__name__}: {e}\n")
            pdf_err_count += 1
            print (f"Outer exception: {e}")

            error_message = (
                f"ERROR | {pdf_path} | page {page} | "
                f"question {q.get('id', 'UNKNOWN')} | "
                f"{type(e).__name__}: {e}"
            )
    
            print(error_message)
            traceback.print_exc()
            raise


116 pages. pdf err: 3, page err: 0, q err: 97 115 processed.  99.1% completed.  Elapsed: 34.0 min. Est. remaining: 0.3 min
Page 8, Survey 540 OK
Question Q4 OK 
Question Q5a OK 
Question Q5b OK 
Question Q5c OK 
Question Q6 OK 
Question Q7 OK 
Question Q8 OK 
Question Q9 OK 
Question Q10a OK 
Question Q10b OK 
Question Q10c OK 
Question Q11 OK 
Question Q12 OK 
Question Q13 OK 
Question Q14 OK 
Question Q15 OK 
Question Q16 OK 
Question Q17 OK 
Question Q18 OK 
Question Q19 OK 


In [107]:
QUESTIONS[30]

{'id': 'Q27',
 'alignment_box': [0.01, 0.85, 0.49, 0.99],
 'page': 4,
 'type': 'written',
 'inputs': [{'kind': 'written',
   'label': 'Feedback or comments',
   'alignment_box': [-0.001467, 0.871085, 0.456592, 0.979335]}],
 'text': 'Feedback or comments'}